In [1]:
import json
from typing import List
from unstructured.chunking.title import chunk_by_title
from unstructured.partition.pdf import partition_pdf

from langchain_core.documents import Document
from langchain_openai import ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings

load_dotenv() 

c:\ML projects\Multimodal RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
def partition_document(filepath):
    print(f"Partitioning document: {filepath}")

    elements = partition_pdf(
        filename= filepath,
        strategy="hi_res",#most accurate but slower process
        infer_table_structure=True,#keep tables as structured html not jumbled text
        extract_image_block_types=["Image"],#grab images found in the pdf
        extract_image_block_to_payload=True #store images as base64 data
    )
    print(f"extracted {len(elements)} elements")
    return elements

filepath = "C:/ML projects/Multimodal RAG/content/attention.pdf"
elements = partition_document(filepath=filepath)
  

Partitioning document: C:/ML projects/Multimodal RAG/content/attention.pdf


The `max_size` parameter is deprecated and will be removed in v4.26. Please specify in `size['longest_edge'] instead`.


extracted 173 elements


In [3]:
def create_chunks_by_title(elements):
    print("Creating smart chunks")

    chunks = chunk_by_title(
        elements, 
        max_characters=3000, #never exceed 3000 characters per chunk
        new_after_n_chars=2500, #Try to start a new chunk after 2400 characters
        combine_text_under_n_chars=500 #chunks under 500 characters will be combined with neighbours
    )
    print(f"Created {len(chunks)} chunks")
    return chunks


chunks = create_chunks_by_title(elements=elements)

Creating smart chunks
Created 21 chunks


In [5]:
def separate_content_types(chunks):
    content_data = {
        "text": chunks.text,
        "tables": [],
        "images": [],
        "types": ["text"]
    }

    if hasattr(chunks, "metadata") and hasattr(chunks.metadata, "orig_elements"):
        for element in chunks.metadata.orig_elements:
            element_type = type(element).__name__

            if element_type == "Table":
                content_data["types"].append("table")
                table_html = getattr(element.metadata, "text_as_html", element.text)
                content_data["tables"].append(table_html)

            elif element_type == "Image":
                if hasattr(element, "metadata") and hasattr(element.metadata, "image_base64"):
                    content_data["types"].append("image")
                    content_data["images"].append(element.metadata.image_base64)

    content_data["types"] = list(set(content_data["types"]))
    return content_data


def create_ai_enhanced_summary(text, tables, images):
    try:
        llm = ChatOpenAI(
            model="nvidia/nemotron-nano-12b-v2-vl:free",
            base_url="https://openrouter.ai/api/v1",
            temperature=0.4
        )

        prompt_text = f"""You are creating a searchable description for document content retrival.

Content to analyze:
text content: {text}
"""

        if tables:
            prompt_text += "tables\n"
            for i, table in enumerate(tables):
                prompt_text += f"Table {i+1}:\n{table}\n\n"

        prompt_text += """
Your task:
Generate a comprehensive, searchable description that covers:
1. key facts, numbers and data point from the tables.
2. main topics and concepts discussed.
3. questions this content could answer.
4. visual content analysis [charts, diagrams, images]
5. alternative search terms users might use

Make it searchable and detailed. Prioritize findability over brevity.

SEARCHABLE DESCRIPTION:
"""

        message_content = [
            {"type": "text", "text": prompt_text}
        ]

        for image_base64 in images:
            message_content.append({
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{image_base64}"
                }
            })

        message = HumanMessage(content=message_content)
        response = llm.invoke([message])

        return response.content if response.content else text

    except Exception as e:
        print(f"AI summary failed: {e}")
        summary = f"{text[:300]}..."
        if tables:
            summary += f"[contains {len(tables)} table(s)]"
        if images:
            summary += f"[contains {len(images)} image(s)]"
        return summary


def summarize_chunks(chunks):
    print("Processing chunks with AI summaries...")
    langchain_documents = []
    total_chunks = len(chunks)

    for i, chunk in enumerate(chunks):
        current_chunk = i + 1
        print(f"Processing chunk {current_chunk}/{total_chunks}")

        content_data = separate_content_types(chunk)

        print(f"content found {content_data['types']}")
        print(f"Tables: {len(content_data['tables'])}, Images: {len(content_data['images'])}")

        if content_data["tables"] or content_data["images"]:
            print("...Creating AI summary for mixed content...")
            try:
                enhanced_summary = create_ai_enhanced_summary(
                    content_data["text"],
                    content_data["tables"],
                    content_data["images"]
                )

                enhanced_summary = enhanced_summary or content_data["text"]

                print("AI summary creation successful")
                print(f"Enhanced content preview: {enhanced_summary[:200]}...")

            except Exception as e:
                print(f"AI summary generation failed: {e}")
                enhanced_summary = content_data["text"]

        else:
            print("Using raw text (no tables/images)")
            enhanced_summary = content_data["text"]

        doc = Document(
            page_content=enhanced_summary,
            metadata={
                "original_content": json.dumps({
                    "raw_text": content_data["text"],
                    "tables_html": content_data["tables"],
                    "image_base64": content_data["images"]
                })
            }
        )

        langchain_documents.append(doc)

    print(f"Processed {len(langchain_documents)} chunks")
    return langchain_documents


processed_chunks = summarize_chunks(chunks=chunks)
print(type(processed_chunks), processed_chunks[:1])

Processing chunks with AI summaries...
Processing chunk 1/21
content found ['text']
Tables: 0, Images: 0
Using raw text (no tables/images)
Processing chunk 2/21
content found ['text']
Tables: 0, Images: 0
Using raw text (no tables/images)
Processing chunk 3/21
content found ['text']
Tables: 0, Images: 0
Using raw text (no tables/images)
Processing chunk 4/21
content found ['text']
Tables: 0, Images: 0
Using raw text (no tables/images)
Processing chunk 5/21
content found ['image', 'text']
Tables: 0, Images: 1
...Creating AI summary for mixed content...
AI summary creation successful
Enhanced content preview: **SEARCHABLE DESCRIPTION**

---

### **1. Key Facts, Numbers, and Data Points**  
- **Encoder**:  
  - Composed of **N = 6 identical layers**.  
  - Each layer contains **2 sub-layers**:  
    1. **Mu...
Processing chunk 6/21
content found ['image', 'text']
Tables: 0, Images: 2
...Creating AI summary for mixed content...
AI summary creation successful
Enhanced content preview: **SEA

In [6]:
def export_chunks_to_json(chunks, filename = "chunks_export.json"):
    export_data = []

    for i,doc in enumerate(chunks):
        chunk_data = {
            "chunk_id": i+1,
            "enhanced_content": doc.page_content,
            "metadata": {
                "original_content": json.loads(doc.metadata.get("original_content", "{}"))
            }
        }
        export_data.append(chunk_data)

    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(export_data, f, indent=2, ensure_ascii=False)


    print(f"Exported {len(export_data)} chunks to {filename}")
    return export_data

json_data = export_chunks_to_json(processed_chunks)


Exported 21 chunks to chunks_export.json


In [10]:
def create_vector_store(documents, persist_directory = "db/chroma_db"):
    print("...Creating embeddings and storing in vector store...")

    embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

    print("-----Creating vector store-----")
    vector_store = Chroma.from_documents(
        documents=documents,
        embedding= embedding_model,
        persist_directory= persist_directory,
        collection_metadata={"hnsw:space": "cosine"}
    )
    print("Finished creating vector store")

    print(f"Created vector store at directory: {persist_directory}")
    return vector_store

db = create_vector_store(processed_chunks)

...Creating embeddings and storing in vector store...
-----Creating vector store-----
Finished creating vector store
Created vector store at directory: db/chroma_db


In [13]:
def run_full_ingestion_pipeline(pdf_path):
    print("Starting full RAG ingestion pipeline")
    print("="*50)

    #Step 1: Partition
    elements = partition_document(pdf_path)

    #Step 2: Chunk
    chunks = create_chunks_by_title(elements)

    #Step 3: AI summarization
    summarized_chunks = summarize_chunks(chunks)

    #Step 4: Vector store
    db = create_vector_store(summarized_chunks, persist_directory="dbv2/chroma_db")

    print("pipeline completed succcessfully")
    return db

In [14]:
db = run_full_ingestion_pipeline("./content/attention.pdf")


Starting full RAG ingestion pipeline
Partitioning document: ./content/attention.pdf
extracted 173 elements
Creating smart chunks
Created 21 chunks
Processing chunks with AI summaries...
Processing chunk 1/21
content found ['text']
Tables: 0, Images: 0
Using raw text (no tables/images)
Processing chunk 2/21
content found ['text']
Tables: 0, Images: 0
Using raw text (no tables/images)
Processing chunk 3/21
content found ['text']
Tables: 0, Images: 0
Using raw text (no tables/images)
Processing chunk 4/21
content found ['text']
Tables: 0, Images: 0
Using raw text (no tables/images)
Processing chunk 5/21
content found ['image', 'text']
Tables: 0, Images: 1
...Creating AI summary for mixed content...
AI summary creation successful
Enhanced content preview: **SEARCHABLE DESCRIPTION****1. Key Facts, Numbers, and Data Points**  
- **Encoder Stack**:  
  - Composed of **N = 6 identical layers**.  
  - Each layer contains **two sub-layers**:  
    1. **Multi...
Processing chunk 6/21
content foun

In [17]:
query = "What are the two main components of transformer architecture?"
retriever= db.as_retriever(search_kwargs = {"k":3})
chunks = retriever.invoke(query)

export_chunks_to_json(chunks, "rag_result.json")

def generate_final_answer(chunks,query):
    try:
        llm = ChatOpenAI(model="mistralai/mistral-small-3.1-24b-instruct:free", base_url="https://openrouter.ai/api/v1", temperature=0)

        prompt_text = f'''Based on the following documents, answer the question: {query}
        CONTENT TO ANALYZE: '''

        for i, chunk in enumerate(chunks):
            prompt_text += f"-----Document {i+1}-----"

            if "original_content" in chunk.metadata:
                original_data = json.loads(chunk.metadata['original_content'])

                raw_text = original_data.get("raw_text", "")
                if raw_text:
                    prompt_text += f"TEXT: \n{raw_text}\n\n"

                tables_html = original_data.get("tables_html", [])
                if tables_html:
                    prompt_text += f"TABLES: "
                    for j, table in enumerate(tables_html):
                        prompt_text += f"table {j+1}: {table}"
                
            prompt_text += "\n"

        prompt_text += '''
        Please provide a clear, comprehensive answer using the text, tables, images above. If the document doesnt contain suficient answer to the question, 
        just say "I don't have enough information to answer the question.

        ANSWER:
        '''
        message_content = [{"type": "text", "text": prompt_text}]

        for chunk in chunks:
            if "original_content" in chunk.metadata:
                original_data = json.loads(chunk.metadata["original_content"])
                images_base64 = original_data.get("image_base64", [])

            for image_base64 in images_base64:
                message_content.append({
                    "type": "image_url",
                    "image_url": {"url": f"data:image/jpeg;base64, {image_base64}"}
                })

        message = HumanMessage(content=message_content)
        response = llm.invoke([message])

        return response.content
    
    except Exception as e:
        print(f"Answer generation failed: {e}")
        return "Sorry I encountered an error while generating the answer"
    
final_answer = generate_final_answer(chunks,query)
print(final_answer)



Exported 3 chunks to rag_result.json
The two main components of the transformer architecture are the encoder and the decoder. Both the encoder and the decoder use stacked self-attention and point-wise, fully connected layers. The encoder maps an input sequence of symbol representations to a sequence of continuous representations, while the decoder generates an output sequence of symbols one element at a time, consuming the previously generated symbols as additional input when generating the next.
